In [1]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

class AgentRouter:
    def __init__(self):
        # 🟢 UPDATED: Optimized to lock down exact keyword sub-word combinations
        self.vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=1)
        self.model = LogisticRegression(C=2.0)
        self._train_router()

    def _train_router(self):
        # Default specialized routing matrix data
        training_data = [
            {"text": "How do I pay my bill?", "label": "billing"},
            {"text": "Where is my invoice?", "label": "billing"},
            {"text": "My app keeps crashing on login", "label": "technical"},
            {"text": "Reset my password link not working", "label": "technical"},
            {"text": "What are the specs of the product?", "label": "product"},
            {"text": "Compare pricing tiers", "label": "product"},
            {"text": "I want to file a formal complaint against service", "label": "complaint"},
            {"text": "This service is horrible, agent was rude", "label": "complaint"},
            {"text": "What is the capital of India?", "label": "faq"},
            {"text": "Explain how photosynthesis works", "label": "faq"}
        ]
        
        # Try importing from local Banking77 dataset to enrich billing/technical routing matrices
        banking_path = "datasets/Banking77_Intent_classification_Dataset"
        if os.path.exists(banking_path):
            try:
                for f in os.listdir(banking_path):
                    if f.endswith('.csv'):
                        df = pd.read_csv(os.path.join(banking_path, f), nrows=200)
                        for text in df.iloc[:, 0].dropna().tolist():
                            training_data.append({"text": text, "label": "billing"})
            except:
                pass

        df_train = pd.DataFrame(training_data)
        X = self.vectorizer.fit_transform(df_train["text"])
        y = df_train["label"]
        self.model.fit(X, y)

    def route_query(self, query: str):
        vec = self.vectorizer.transform([query])
        pred = self.model.predict(vec)[0]
        prob = max(self.model.predict_proba(vec)[0])
        return pred, prob